# GEMAMX v2 Romero - Estructuras con GPU en Google Colab

Esta libreta corre la parte pesada de prediccion estructural de `v2` usando `Boltz` en un runtime con GPU.

Alcance de esta libreta:
- leer `v2/data/fasta_sequences.json`
- predecir WT y MUT con GPU
- guardar resultados en `v2/data/structures`
- actualizar `v2/data/manifest_structures.json` y `v2/data/structure_jobs.json`


## Antes de correr

1. Abre esta libreta en Colab.
2. Cambia el runtime a **GPU**.
3. Asegurate de tener el repo en Google Drive en `MyDrive/GEMAMX`.
4. Asegurate de que ya existan estos archivos generados localmente:
   - `v2/data/top_variants.json`
   - `v2/data/fasta_sequences.json`
   - `v2/data/fasta/*.fasta`


In [ ]:
from google.colab import drive
from pathlib import Path
import os

drive.mount('/content/drive')
ROOT = Path('/content/drive/MyDrive/GEMAMX')
assert ROOT.exists(), f'No existe el repo en Drive: {ROOT}'
os.chdir(ROOT)
print('Workspace:', ROOT)


In [ ]:
!nvidia-smi
%pip install -q boltz
!boltz --version


In [ ]:
import json
from pathlib import Path

fasta_index = ROOT / 'v2' / 'data' / 'fasta_sequences.json'
assert fasta_index.exists(), f'No existe: {fasta_index}'
entries = json.loads(fasta_index.read_text(encoding='utf-8'))
print('FASTA pairs listos:', len(entries))
print('Ejemplos:')
for item in entries[:5]:
    print(' -', item['gene'], item['protein_change'], 'pos', item['mutation_pos'])


In [ ]:
!python v2/scripts/predict_structures_romero_boltz.py --only-missing


In [ ]:
!python v2/scripts/scan_boltz_v2.py
!python v2/scripts/calc_structural_metrics_romero.py


In [ ]:
from pathlib import Path
import zipfile

zip_path = ROOT / 'v2' / 'data' / 'romero_structures_gpu_export.zip'
targets = [
    ROOT / 'v2' / 'data' / 'structures',
    ROOT / 'v2' / 'data' / 'manifest_structures.json',
    ROOT / 'v2' / 'data' / 'structure_jobs.json',
    ROOT / 'v2' / 'data' / 'structural_metrics.json',
]
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for target in targets:
        if target.is_dir():
            for item in target.rglob('*'):
                if item.is_file():
                    zf.write(item, item.relative_to(ROOT))
        elif target.exists():
            zf.write(target, target.relative_to(ROOT))
print('ZIP listo:', zip_path)
